# Trabajo Práctico Aprendizaje Automático 2

## Imports

In [ ]:
%load_ext autoreload
%autoreload 2

from src.datos.carga import cargar_wikipedia
from src.datos.preprocesamiento import construir_dataset
from src.datos.analisis import resumen_etiquetas

## Tokenizador

Utilizamos el código proporcionado.

In [2]:
from transformers import BertTokenizer, BertModel
import torch

model_name = "bert-base-multilingual-cased"
tokenizer = BertTokenizer.from_pretrained(model_name)
model = BertModel.from_pretrained(model_name)
model.eval()

def get_multilingual_token_embedding(token: str):
    """
    Devuelve el embedding (estático) para el token.
    """
    token_id = tokenizer.convert_tokens_to_ids(token)

    if token_id is None or token_id == tokenizer.unk_token_id:
        print(f"❌El token '{token}' no pertenece al vocabulario de multilingual BERT.")
        return None

    embedding_vector = model.embeddings.word_embeddings.weight[token_id]

    print(f"✅Token: '{token}' | ID: {token_id}")
    print(f"Embedding shape: {embedding_vector.shape}")
    return embedding_vector

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/996k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.96M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  714MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


## Dataset

Primero tenemos que conseguir textos para llevar a cabo la tarea. Vamos a utilizar un dataset de Wikipedia (por ahora) después lo podemos cambiar.

(Agregado giani) sumo que opensubtitles puede estar bueno para tener mas instancias de (?) y (¿). Habria que mezclar ambos

In [ ]:
# aleatorio=True toma artículos al azar en vez de los primeros 100 seguidos
muestra_textos = cargar_wikipedia(n_textos=100, min_len=100, aleatorio=False)

print(f"Se descargaron {len(muestra_textos)} textos de muestra.")

# Mostramos el primer ejemplo crudo
print("\nEjemplo de texto crudo descargado:")
print(muestra_textos[0][:250] + "...")

Una vez que bajamos los parráfos de wikipedia, los vamos a procesar para armar el dataset que utilizaremos luego.

### Funciones de procesamiento

Están en `src/datos/preprocesamiento.py`:
- `normalizar_signos`: reemplaza o elimina los signos que no son etiquetas (`REEMPLAZOS`).
- `seccionar_palabra`: separa una palabra cruda en puntuación inicial, sub-palabras limpias y puntuación final.
- `determinar_capitalizacion`: clase de capitalización (0, 1, 2, 3) según la consigna.
- `procesar_parrafo_a_chunks`: tokeniza con BERT, asigna las etiquetas a cada sub-token y parte en chunks.
- `construir_dataset`: aplica lo anterior a todos los textos y concatena.

Reemplaza la extracción de puntuación que estaba dentro de `procesar_parrafo_a_chunks`. Antes solo se miraba el primer y el último carácter, así que en casos como `"¿Vienes?"` o `(Andorra),` se perdía la etiqueta, y los demás signos quedaban como tokens. Ahora se limpian todos los signos de los bordes, las palabras con guion se separan (`Castilla-La` → `Castilla La`) y se borran los signos internos (`EE.UU` → `EEUU`).

Aplicamos sobre la lista de parráfos bajados.

In [ ]:
df_dataset_final = construir_dataset(muestra_textos, tokenizer, chunk_size=32)

print("Muestra del Dataset de Entrenamiento generado por Chunks:")
print(df_dataset_final.head(15))

Veamos cuantas instacias tenemos

In [ ]:
resumen_etiquetas(df_dataset_final)